# 03 — Base de logradouros com CEP (Correios × Geosanja + CNEFE automático)

**Objetivo:** montar a base de logradouros com CEP **sem revisão manual**, em dois processos, e marcar em cada segmento qual processo deu o CEP e com que confiança.

| Passo | Processo (`cep_origem`) | O que faz |
|---|---|---|
| 1 | `correios_geosanja:<método>` | Segmentos `pareado` na etapa 02 (um CEP só) ficam com o CEP dos Correios |
| 2 | (auditoria) | Se a **mediana** da distância dos pontos do CNEFE de um CEP pareado até os seus segmentos passar de `mediana_suspeita_m`, o CEP é **retirado** desses segmentos e eles vão para o passo 3 (`cnefe_motivo = cep_suspeito:...`) |
| 3 | `cnefe_segmento` | Os pontos do CNEFE **votam** no CEP de **todos** os segmentos. Segmentos abertos (`multi_cep`, `ambiguo`, `sem_cep`, `sem_nome` e os liberados no passo 2) recebem o vencedor direto, com qualquer confiança |
| 3 | (correção dos Correios) | Segmento com CEP dos Correios: vencedor **igual** → mantém e sobe para confiança `alta` (`cnefe_motivo = confirmado_cnefe`); **diferente** com confiança `alta`/`media` → troca (`correios_divergente:<cep dos correios>`); diferente com confiança `baixa` → mantém (`correios_divergente_fraco:<cep do cnefe>`) |
| — | `correios_geosanja:multi_cep_sem_cnefe` | `multi_cep` sem nenhum ponto do CNEFE: mantém os CEPs dos Correios (confiança baixa) |

**Ligação ponto → segmento (com filtro de nome):** cada local do CNEFE vai para **um** segmento, nesta ordem de prioridade:
1. segmento com **nome compatível** com a rua do CNEFE, a até `dist_perto_m`;
2. segmento **sem nome**, a até `dist_perto_m`;
3. segmento com nome compatível, a até `dist_busca_m`;
4. segmento sem nome, a até `dist_busca_m`.

Isso evita que o ponto de esquina vote na rua transversal.

**Confiança (`cep_confianca`):**

| Valor | Quando |
|---|---|
| `alta` | Correios por nome exato · CNEFE com ≥ `min_pontos` locais e vencedor com ≥ `min_participacao` dos votos |
| `media` | Correios por iniciais ou fuzzy · CNEFE com vencedor ≥ `participacao_baixa`, sem empate |
| `baixa` | CNEFE com vencedor abaixo de `participacao_baixa` ou empate · `multi_cep` sem pontos do CNEFE |

**Segmento sem nome:** se o CNEFE der o CEP, o nome também vem do CNEFE (`nome_origem = cnefe`; nome original em `nome_geosanja`).

**Regra do editado:** se `02_dados_editados\logradouros_cep.gpkg` **já existe**, ele **não é alterado** (são as suas edições). A versão automática vai para `03_processamento\03_base_logradouros_cep\logradouros_cep_automatico.gpkg`, só para comparação. Para refazer do zero, apague a base editada e rode esta etapa.

**Saídas:**
- `02_dados_editados\logradouros_cep.gpkg` (camada `logradouros_cep`) — só na primeira vez;
- `03_processamento\03_base_logradouros_cep\cnefe_locais.parquet` — locais do CNEFE e o segmento de cada um (usado na etapa 05);
- `03_processamento\03_base_logradouros_cep\pontos_cnefe.gpkg` — os mesmos locais, para ver no QGIS;
- `03_processamento\03_base_logradouros_cep\relatorio_base_cnefe.xlsx` — resumo, CEPs trocados e segmentos preenchidos pelo CNEFE.

**Estrutura:** 1 Preparação · 2 Funções · 3 Geosanja e etapa 02 · 4 Passo 1 (Correios) · 5 CNEFE · 6 Ligar pontos aos segmentos · 7 Passo 2 (suspeitos) · 8 Passo 3 (votação) · 9 Aplicar na base · 10 Exportação · 11 Resumo

## 1. Preparação e configuração
Parâmetros do CNEFE na seção `parametros.cnefe` do `config.yaml`.

In [ ]:
# --- Preparação padrão: acha a raiz do projeto, lê o config.yaml e liga o log (tela + arquivo) -------------
import sys                                            # para o Python achar as funções comuns (scripts/src)
from pathlib import Path                              # caminhos de arquivos
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())   # sobe pastas até achar o config.yaml
sys.path.insert(0, str(RAIZ / "scripts"))             # libera o "from src import ..."
from src import config, registro                      # funções comuns do padrão de projetos
cfg = config.carregar(RAIZ)                           # lê o config.yaml (caminhos e parâmetros)
ETAPA = "03_base_logradouros_cep"                     # nome desta etapa (= nome do notebook)
log = registro.iniciar_log(ETAPA, cfg)                # log na tela e em 03_processamento/logs/
PASTA_SAIDA = cfg.pasta_etapa(ETAPA)                  # 03_processamento/<ETAPA> (criada se não existir)
import numpy as np                    # contas com colunas
import pandas as pd                   # tabelas
import pyogrio                        # leitura/escrita de arquivos geográficos
import geopandas as gpd               # tabelas com geometria
from rapidfuzz import fuzz            # comparação de textos "parecidos"

# --- Entradas (config.yaml) ----------------------------------------------------------------------------
ARQ_GEOSANJA = cfg.caminho("brutos.geosanja")                       # camada original, com todos os campos
CAMADA_ORIGINAL = cfg.camada("brutos.geosanja")
ARQ_ETAPA2_GPKG = cfg.caminho("processamento.pareamento")           # resultado automático da etapa 02
CAMADA_ETAPA2 = cfg.camada("processamento.pareamento")
ARQ_PONTE = cfg.caminho("processamento.ponte")                      # CEPs dos correios pareados (etapa 02)
ARQ_ORFAOS = cfg.caminho("processamento.orfaos")                    # CEPs dos correios sem segmento (etapa 02)
ARQ_CNEFE = cfg.caminho("brutos.cnefe")                             # cnefe do município

# --- Saídas ------------------------------------------------------------------------------------------------
ARQ_BASE = cfg.caminho("editados.logradouros_cep")                  # base oficial (02_dados_editados)
CAMADA_BASE = cfg.camada("editados.logradouros_cep")
ARQ_LOCAIS = cfg.caminho("processamento.cnefe_locais")              # locais do cnefe (lido pela etapa 05)
ARQ_PONTOS = PASTA_SAIDA / "pontos_cnefe.gpkg"                      # locais do cnefe para o QGIS
ARQ_EXCEL = PASTA_SAIDA / "relatorio_base_cnefe.xlsx"               # relatório desta etapa

# --- Parâmetros --------------------------------------------------------------------------------------------
CRS_METRICO = cfg.projeto("crs_metrico")                            # geosanja (metros)
CRS_GRAUS = cfg.projeto("crs_graus")                                # latitude/longitude do cnefe
DIST_PERTO_M = cfg.parametro("cnefe.dist_perto_m")                  # ponto a até isso da linha = "na linha"
DIST_BUSCA_M = cfg.parametro("cnefe.dist_busca_m")                  # distância máxima ponto -> segmento
MEDIANA_SUSPEITA_M = cfg.parametro("cnefe.mediana_suspeita_m")      # mediana acima disso = CEP dos correios trocado
MIN_PONTOS = cfg.parametro("cnefe.min_pontos")                      # mínimo de locais para confiança alta (e para julgar suspeito)
MIN_PARTICIPACAO = cfg.parametro("cnefe.min_participacao")          # participação mínima para confiança alta
PARTICIPACAO_BAIXA = cfg.parametro("cnefe.participacao_baixa")      # abaixo disso a confiança é baixa
LIMIAR_NOME = cfg.parametro("cnefe.limiar_nome")                    # similaridade mínima (0-100) entre o nome do cnefe e o do segmento
COLUNAS_CNEFE = ["CEP", "NOM_TIPO_SEGLOGR", "NOM_TITULO_SEGLOGR", "NOM_SEGLOGR", "LATITUDE", "LONGITUDE"]
STATUS_ABERTOS = ["multi_cep", "ambiguo", "sem_cep", "sem_nome"]    # status da etapa 02 que vão para o CNEFE

log.info("Configuração carregada!")
for _nome, _arq in [("geosanja original", ARQ_GEOSANJA), ("Pareamento (etapa 02)", ARQ_ETAPA2_GPKG), ("Ponte", ARQ_PONTE),
                    ("Órfãos", ARQ_ORFAOS), ("cnefe", ARQ_CNEFE), ("Base editada", ARQ_BASE)]:
    log.info("%s: %s | existe: %s", _nome, _arq.name, _arq.exists())
log.info("Limites: perto %s m | busca %s m | mediana suspeita %s m | mín. %s locais | participação alta %.0f%% / baixa %.0f%% | nome %s",
         DIST_PERTO_M, DIST_BUSCA_M, MEDIANA_SUSPEITA_M, MIN_PONTOS, MIN_PARTICIPACAO * 100, PARTICIPACAO_BAIXA * 100, LIMIAR_NOME)
if ARQ_BASE.exists():
    log.warning("A base editada já existe: ela NÃO será alterada. A versão automática irá só para %s.", PASTA_SAIDA.name)

## 2. Funções auxiliares
- `nome_chave`: nome de rua comparável (maiúsculas, sem acento, sem preposição e **sem o tipo**);
- `nomes_compativeis`: diz se o nome da rua no CNEFE e o nome do segmento são a mesma rua;
- `texto_votos`: resume a votação de um segmento (`12345678: 7 | 12345679: 2`).

In [ ]:
from src.validacao import validar_tabela                 # bloco padrão de validação
from src.texto import normalizar, tirar_tipo             # normalização de nomes de logradouro
from src.geo import para_gpkg, checar_crs                # tipos compatíveis com GeoPackage e conferência de CRS
from src.planilha import formatar_excel                  # formatação das abas do Excel


def nome_chave(serie: pd.Series) -> pd.Series:
    """Nome de rua comparável: normalizado (src.texto) e sem o tipo (RUA, AVENIDA...). Nulo vira vazio."""
    return normalizar(serie).map(tirar_tipo)


def nomes_compativeis(a: str, b: str) -> bool:
    """True se os dois nomes (já em nome_chave) são a mesma rua.

    Vale se a ordem das palavras quase não importa (token_sort_ratio >= LIMIAR_NOME)
    ou se um nome está contido no outro (token_set_ratio = 100) e o menor tem pelo menos 2 palavras
    (ex.: 'JOAO SILVA' x 'JOAO DA SILVA PEREIRA'; evita casar nomes de 1 palavra como 'A' ou 'B').
    """
    if fuzz.token_sort_ratio(a, b) >= LIMIAR_NOME:
        return True
    menor = min(len(a.split()), len(b.split()))
    return menor >= 2 and fuzz.token_set_ratio(a, b) == 100


def texto_votos(d: pd.DataFrame) -> str:
    """Texto curto da votação de um segmento: 'cep: nº de locais', do maior para o menor (até 5 CEPs)."""
    return " | ".join(f"{c}: {k}" for c, k in zip(d["cep"].head(5), d["n_locais"].head(5)))


# --- Teste rápido do filtro de nome (confira se faz sentido) ---------------------------------------------
_t = pd.DataFrame({"cnefe": ["DOUTOR NELSON DAVILA", "JOAO SILVA", "SAO JOAO", "A"],
                   "segmento": ["Doutor Nelson D'Ávila", "João da Silva Pereira", "Avenida São João", "B"]})
_t["a"], _t["b"] = nome_chave(_t["cnefe"]), nome_chave(_t["segmento"])
_t["compativel"] = [nomes_compativeis(a, b) for a, b in zip(_t["a"], _t["b"])]
log.info("Teste do filtro de nome:\n%s", _t.to_string())
log.info("Funções auxiliares definidas!")

## 3. Geosanja original e resultado da etapa 02
Lê a camada original do Geosanja (todos os campos + geometria) e junta, pelo `id`, o status e os CEPs da etapa 02. Também monta o **universo de CEPs dos Correios** (pareados + órfãos): só esses CEPs podem ser gravados nos segmentos.

In [ ]:
log.info("Carregando a camada original do geosanja...")
orig = pyogrio.read_dataframe(ARQ_GEOSANJA, layer=CAMADA_ORIGINAL)
checar_crs(orig, CRS_METRICO, "geosanja")                       # para se o CRS não for o esperado
n_orig = len(orig)
for col in ["id", "tipo", "nome", "loteamento"]:
    assert col in orig.columns, f"Campo '{col}' não existe na camada original"
colunas_originais = [c for c in orig.columns if c != "geometry"]
log.info("Feições: %d | CRS: %s | campos originais: %s", n_orig, orig.crs, colunas_originais)
validar_tabela("geosanja original", orig, chave=["id"], obrigatorias=["id"])

log.info("Carregando o resultado da etapa 02...")
e2 = pyogrio.read_dataframe(ARQ_ETAPA2_GPKG, layer=CAMADA_ETAPA2, read_geometry=False,
                            columns=["id", "status", "cep", "ceps_candidatos", "metodo"])
e2 = e2.rename(columns={"status": "status_cep", "cep": "cep_e2", "ceps_candidatos": "cand_e2", "metodo": "metodo_e2"})
validar_tabela("Etapa 02 (status por segmento)", e2, chave=["id"], obrigatorias=["id", "status_cep"])

base = orig.merge(e2, on="id", how="left", validate="one_to_one")          # mantém a geometria do geosanja
assert len(base) == n_orig, "A junção mudou o número de feições!"
for col in ["status_cep", "cep_e2", "cand_e2", "metodo_e2", "tipo", "nome"]:
    base[col] = base[col].astype(object).where(base[col].notna(), "").astype(str).str.strip()   # nulo vira texto vazio
sem_e2 = int((base["status_cep"] == "").sum())
(log.info if sem_e2 == 0 else log.error)("Segmentos sem resultado da etapa 02: %d", sem_e2)
log.info("Status (etapa 02):\n%s", base["status_cep"].value_counts().to_string())

log.info("Montando o universo de CEPs dos correios...")
ponte = pd.read_parquet(ARQ_PONTE, columns=["cep"])
orfaos = pd.read_csv(ARQ_ORFAOS, sep=";", dtype=str, encoding="utf-8-sig", keep_default_na=False)
universo = set(ponte["cep"].astype(str)) | set(orfaos["cep"])                # CEPs de logradouro que existem nos correios
log.info("CEPs nos correios: %d", len(universo))
log.info("Entradas carregadas!")

## 4. Passo 1 — CEP dos Correios
- `pareado` → `cep` = CEP da etapa 02, `cep_origem = correios_geosanja:<método>`, confiança `alta` (nome exato) ou `media` (iniciais/fuzzy);
- `multi_cep`, `ambiguo`, `sem_cep`, `sem_nome` → ficam abertos para o CNEFE (`cnefe_motivo` = o status);
- `cep_correios` guarda o que os Correios deram (CEP único ou lista `a|b`), para comparar depois.

In [ ]:
log.info("Passo 1: aplicando o CEP dos correios...")
st = base["status_cep"]                                                                   # atalho
base["cep_correios"] = np.select([st == "pareado", st == "multi_cep"],
                                 [base["cep_e2"], base["cand_e2"]], default="")           # o que os correios deram
base["ceps_candidatos"] = np.where(st == "ambiguo", base["cand_e2"], "")                 # candidatos dos ambíguos
base["cep"] = np.where(st == "pareado", base["cep_e2"], "")                              # CEP final (por enquanto só os pareados)
base["cep_origem"] = np.where(st == "pareado", "correios_geosanja:" + base["metodo_e2"], "")
base["cep_confianca"] = np.where(st == "pareado",
                                 np.where(base["metodo_e2"].str.startswith("exato"), "alta", "media"), "")
base["cnefe_motivo"] = np.where(st.isin(STATUS_ABERTOS), st, "")                         # por que o segmento vai para o CNEFE

# --- Nome comparável do segmento e marca de "sem nome" (mesma regra da etapa 02) ---------------------------------
base["nome_key"] = nome_chave(base["nome"])
base["seg_sem_nome"] = (base["nome_key"] == "") | base["nome_key"].str.match(r"^SEM NOME\b")

log.info("Segmentos com CEP dos correios: %d | abertos para o CNEFE: %d", int((base["cep"] != "").sum()),
         int((base["cnefe_motivo"] != "").sum()))
log.info("Passo 1 concluído!")

## 5. CNEFE → locais
Lê só as colunas necessárias, mantém endereços com CEP de 8 dígitos e coordenada, e junta os de **mesma coordenada e mesmo CEP** num **local** (um prédio com 80 apartamentos conta 1 voto). `n_enderecos` guarda quantos eram. A rua do local é `título + nome` do CNEFE (sem o tipo).

In [ ]:
log.info("Carregando o cnefe...")
def ler_cnefe(codificacao):
    """Lê o cnefe (separador ';') só com as colunas necessárias, tudo como texto."""
    return pd.read_csv(ARQ_CNEFE, sep=";", dtype=str, usecols=COLUNAS_CNEFE, encoding=codificacao, keep_default_na=False)

try:
    cnefe = ler_cnefe("utf-8")
except UnicodeDecodeError:
    log.warning("Arquivo não está em UTF-8; relendo como Latin-1.")
    cnefe = ler_cnefe("latin-1")
n_cnefe = len(cnefe)
log.info("Endereços lidos: %d!", n_cnefe)

log.info("Padronizando o cnefe...")
cnefe["cep"] = cnefe["CEP"].str.replace(r"\D", "", regex=True)                                   # só dígitos
cnefe["lat"] = pd.to_numeric(cnefe["LATITUDE"].str.replace(",", ".", regex=False), errors="coerce")
cnefe["lon"] = pd.to_numeric(cnefe["LONGITUDE"].str.replace(",", ".", regex=False), errors="coerce")
cnefe["tipo_cnefe"] = cnefe["NOM_TIPO_SEGLOGR"].str.strip()                                       # RUA, AVENIDA...
cnefe["nome_cnefe"] = (cnefe["NOM_TITULO_SEGLOGR"] + " " + cnefe["NOM_SEGLOGR"]).str.split().str.join(" ")   # título + nome
validos = (cnefe["cep"].str.len() == 8) & cnefe["lat"].notna() & cnefe["lon"].notna()
log.info("Com CEP e coordenada: %d (%.1f%%)", int(validos.sum()), validos.mean() * 100)

log.info("Agrupando em locais (mesma coordenada e mesmo CEP)...")
locais = (cnefe[validos].groupby(["cep", "lat", "lon"], as_index=False)
          .agg(n_enderecos=("cep", "size"), tipo_cnefe=("tipo_cnefe", "first"), nome_cnefe=("nome_cnefe", "first")))
locais["loc_id"] = np.arange(len(locais), dtype="int64")                                          # identificador do local
locais["nome_key_cnefe"] = nome_chave(locais["nome_cnefe"])                                       # nome comparável
locais = gpd.GeoDataFrame(locais, geometry=gpd.points_from_xy(locais["lon"], locais["lat"]), crs=CRS_GRAUS).to_crs(CRS_METRICO)
del cnefe                                                                                         # libera memória
log.info("Locais: %d | CEPs distintos no cnefe: %d | locais sem nome de rua: %d", len(locais), locais["cep"].nunique(),
         int((locais["nome_key_cnefe"] == "").sum()))
validar_tabela("Locais do cnefe", locais, chave=["loc_id"], obrigatorias=["cep", "loc_id"])
log.info("cnefe carregado!")

## 6. Ligar cada local a um segmento (com filtro de nome)
1. acha **todos** os segmentos a até `dist_busca_m` de cada local;
2. compara o nome da rua do CNEFE com o nome de cada segmento (só os pares distintos de nomes, rápido);
3. escolhe **um** segmento por local, pela prioridade (nome compatível perto → sem nome perto → nome compatível longe → sem nome longe) e, no empate, o mais próximo.

Local sem nome de rua no CNEFE aceita qualquer segmento. Local sem segmento compatível fica `sem_ligacao` (ainda serve para a etapa 05).

In [ ]:
log.info("Procurando segmentos a até %s m de cada local...", DIST_BUSCA_M)
segs = gpd.GeoDataFrame(base[["id"]].rename(columns={"id": "id_seg"}), geometry=base.geometry, crs=base.crs)
area = gpd.GeoDataFrame(locais[["loc_id"]], geometry=locais.geometry.buffer(DIST_BUSCA_M, resolution=4), crs=CRS_METRICO)
cand = gpd.sjoin(area, segs, how="inner", predicate="intersects")                    # local x segmento que encosta no círculo
cand = pd.DataFrame(cand[["loc_id", "id_seg"]]).drop_duplicates().reset_index(drop=True)
log.info("Pares local x segmento candidatos: %d!", len(cand))

log.info("Medindo a distância exata de cada par...")
geom_loc = locais.set_index("loc_id").geometry                                      # loc_id -> ponto
geom_seg = segs.set_index("id_seg").geometry                                        # id_seg -> linha
g_loc = gpd.GeoSeries(geom_loc.loc[cand["loc_id"].to_numpy()].to_numpy(), crs=CRS_METRICO)   # ponto de cada par
g_seg = gpd.GeoSeries(geom_seg.loc[cand["id_seg"].to_numpy()].to_numpy(), crs=CRS_METRICO)   # linha de cada par
cand["dist"] = g_loc.distance(g_seg).to_numpy()                                     # metros
cand = cand[cand["dist"] <= DIST_BUSCA_M].reset_index(drop=True)                    # tira os que ficaram fora do círculo exato
del g_loc, g_seg
log.info("Pares a até %s m: %d!", DIST_BUSCA_M, len(cand))

log.info("Comparando nomes (cnefe x segmento)...")
cand = (cand.merge(pd.DataFrame(locais[["loc_id", "nome_key_cnefe"]]), on="loc_id")
            .merge(pd.DataFrame(base[["id", "nome_key", "seg_sem_nome"]]).rename(columns={"id": "id_seg"}), on="id_seg"))
com_nomes = (cand["nome_key_cnefe"] != "") & ~cand["seg_sem_nome"]                  # pares em que dá para comparar nome
pares_nome = cand.loc[com_nomes, ["nome_key_cnefe", "nome_key"]].drop_duplicates()  # só os pares distintos de nomes
pares_nome["nome_ok"] = [nomes_compativeis(a, b) for a, b in zip(pares_nome["nome_key_cnefe"], pares_nome["nome_key"])]
log.info("Pares distintos de nomes comparados: %d | compatíveis: %d", len(pares_nome), int(pares_nome["nome_ok"].sum()))
cand = cand.merge(pares_nome, on=["nome_key_cnefe", "nome_key"], how="left")
cand["nome_ok"] = cand["nome_ok"].fillna(False).astype(bool) | ((cand["nome_key_cnefe"] == "") & ~cand["seg_sem_nome"])   # sem nome no cnefe: aceita

log.info("Escolhendo um segmento por local...")
perto = cand["dist"] <= DIST_PERTO_M
cand["prioridade"] = np.select(
    [cand["nome_ok"] & perto, cand["seg_sem_nome"] & perto, cand["nome_ok"], cand["seg_sem_nome"]],
    [1, 2, 3, 4], default=0)                                                         # 0 = segmento não serve para este local
lig = (cand[cand["prioridade"] > 0].sort_values(["loc_id", "prioridade", "dist"])
       .drop_duplicates("loc_id")[["loc_id", "id_seg", "dist", "prioridade"]])
lig["ligacao"] = lig["prioridade"].map({1: "nome", 2: "sem_nome", 3: "nome", 4: "sem_nome"})
locais = locais.merge(lig.drop(columns="prioridade"), on="loc_id", how="left")
locais["ligacao"] = locais["ligacao"].fillna("sem_ligacao")
if pd.api.types.is_integer_dtype(base["id"]):
    locais["id_seg"] = locais["id_seg"].astype("Int64")                             # mesmo tipo do id da base
del cand
assert locais["loc_id"].is_unique, "A ligação duplicou locais!"
log.info("Locais por tipo de ligação:\n%s", locais["ligacao"].value_counts().to_string())
log.info("Ligação concluída!")

## 7. Passo 2 — CEPs dos Correios suspeitos
Para cada CEP de segmento **pareado**, mede a distância de cada um dos seus locais no CNEFE até o segmento mais próximo **daquele CEP** (aqui sem filtro de nome: mede onde o CEP está de fato). Se o CEP tem pelo menos `min_pontos` locais e a **mediana** passa de `mediana_suspeita_m`, ele é **retirado** dos seus segmentos, que vão para a votação do passo 3 com `cnefe_motivo = cep_suspeito:<cep> (mediana N m)`.

Com menos de `min_pontos` locais não há evidência suficiente: o CEP dos Correios fica.

In [ ]:
log.info("Passo 2: auditando os CEPs pareados pelos correios...")
pareados = pd.DataFrame(base.loc[base["status_cep"] == "pareado", ["id", "cep"]]).rename(columns={"id": "id_seg"})
pa = pd.DataFrame(locais.loc[locais["cep"].isin(set(pareados["cep"])), ["loc_id", "cep"]])
m = pa.merge(pareados, on="cep").reset_index(drop=True)                             # cada local x cada segmento do seu CEP
g_loc = gpd.GeoSeries(geom_loc.loc[m["loc_id"].to_numpy()].to_numpy(), crs=CRS_METRICO)
g_seg = gpd.GeoSeries(geom_seg.loc[m["id_seg"].to_numpy()].to_numpy(), crs=CRS_METRICO)
m["dist_cep"] = g_loc.distance(g_seg).to_numpy()
del g_loc, g_seg
dist_loc = m.groupby("loc_id", as_index=False).agg(cep=("cep", "first"), dist_cep=("dist_cep", "min"))   # menor distância por local
aud = (dist_loc.groupby("cep", as_index=False)
       .agg(n_locais=("loc_id", "size"), dist_mediana_m=("dist_cep", "median"),
            pct_perto=("dist_cep", lambda s: round((s <= DIST_PERTO_M).mean() * 100, 1))))
aud["dist_mediana_m"] = aud["dist_mediana_m"].round(1)
aud["suspeito"] = (aud["n_locais"] >= MIN_PONTOS) & (aud["dist_mediana_m"] > MEDIANA_SUSPEITA_M)
log.info("CEPs pareados com pontos no cnefe: %d de %d | suspeitos: %d", len(aud), pareados["cep"].nunique(), int(aud["suspeito"].sum()))

# --- Retira o CEP suspeito dos seus segmentos e manda os segmentos para a votação -----------------------------------
mediana = dict(zip(aud["cep"], aud["dist_mediana_m"]))
ceps_suspeitos = set(aud.loc[aud["suspeito"], "cep"])
mask_s = (base["status_cep"] == "pareado") & base["cep"].isin(ceps_suspeitos)
base.loc[mask_s, "cnefe_motivo"] = ("cep_suspeito:" + base.loc[mask_s, "cep"] + " (mediana "
                                    + base.loc[mask_s, "cep"].map(mediana).round(0).astype(int).astype(str) + " m)")
base.loc[mask_s, ["cep", "cep_origem", "cep_confianca"]] = ""
log.info("Segmentos liberados (CEP suspeito retirado): %d", int(mask_s.sum()))
top_s = aud[aud["suspeito"]].sort_values("n_locais", ascending=False).head(10)
if len(top_s):
    log.info("Suspeitos com mais locais:\n%s", top_s.to_string(index=False))
log.info("Passo 2 concluído!")

## 8. Passo 3 — Votação do CNEFE em todos os segmentos
Votam os locais ligados (seção 6) a **qualquer** segmento, com CEP que **existe nos Correios**. Por segmento:
- vencedor = CEP com mais locais (empate: mais endereços, depois menor distância média);
- `participacao` = locais do vencedor ÷ locais válidos;
- confiança `alta`, `media` ou `baixa` (tabela no topo).

O nome do segmento sem nome vem dos locais do CEP vencedor (o nome mais frequente).

In [ ]:
log.info("Passo 3: votação do cnefe em todos os segmentos...")
abertos = set(base.loc[base["cnefe_motivo"] != "", "id"])                      # segmentos sem CEP (recebem qualquer vencedor)
na_votacao = locais["id_seg"].notna()                                            # todo local ligado a um segmento vota
v = pd.DataFrame(locais.loc[na_votacao & locais["cep"].isin(universo),
                            ["loc_id", "cep", "id_seg", "n_enderecos", "dist", "tipo_cnefe", "nome_cnefe"]])
fora = int((na_votacao & ~locais["cep"].isin(universo)).sum())
log.info("Segmentos abertos: %d | locais votando: %d | locais ignorados (CEP fora dos correios): %d", len(abertos), len(v), fora)

cont = (v.groupby(["id_seg", "cep"], as_index=False)
        .agg(n_locais=("loc_id", "nunique"), n_enderecos=("n_enderecos", "sum"), dist_media=("dist", "mean")))
cont["n_votos_validos"] = cont.groupby("id_seg")["n_locais"].transform("sum")
cont = cont.sort_values(["id_seg", "n_locais", "n_enderecos", "dist_media"], ascending=[True, False, False, True])
cont["ordem"] = cont.groupby("id_seg").cumcount()                                  # 0 = vencedor, 1 = segundo...
votos_txt = cont.groupby("id_seg")[["cep", "n_locais"]].apply(texto_votos).rename("cnefe_votos")

venc = cont[cont["ordem"] == 0].merge(
    cont.loc[cont["ordem"] == 1, ["id_seg", "n_locais"]].rename(columns={"n_locais": "n_locais_2o"}), on="id_seg", how="left")
venc["n_locais_2o"] = venc["n_locais_2o"].fillna(0).astype(int)
venc["empate"] = venc["n_locais"] == venc["n_locais_2o"]                           # desempate só pelos endereços/distância
venc["participacao"] = (venc["n_locais"] / venc["n_votos_validos"]).round(3)
venc["confianca"] = np.select(
    [(venc["n_locais"] >= MIN_PONTOS) & (venc["participacao"] >= MIN_PARTICIPACAO) & ~venc["empate"],
     (venc["participacao"] >= PARTICIPACAO_BAIXA) & ~venc["empate"]],
    ["alta", "media"], default="baixa")
venc = venc.merge(votos_txt, left_on="id_seg", right_index=True, how="left")

# --- Nome do CNEFE para o segmento (o mais frequente entre os locais do CEP vencedor) --------------------------------
nomes = (v.merge(venc[["id_seg", "cep"]], on=["id_seg", "cep"])
          .query("nome_cnefe != ''")
          .groupby(["id_seg", "tipo_cnefe", "nome_cnefe"]).size().rename("n").reset_index()
          .sort_values(["id_seg", "n"], ascending=[True, False]).drop_duplicates("id_seg"))
venc = venc.merge(nomes[["id_seg", "tipo_cnefe", "nome_cnefe"]], on="id_seg", how="left")
log.info("Segmentos com vencedor: %d (abertos: %d) | confiança:\n%s", len(venc), int(venc["id_seg"].isin(abertos).sum()),
         venc["confianca"].value_counts().to_string())
validar_tabela("Vencedores por segmento", venc, chave=["id_seg"], obrigatorias=["id_seg", "cep", "confianca"])
log.info("Passo 3 concluído!")

## 9. Aplicar na base
- segmento aberto **com** vencedor → `cep` = vencedor, `cep_origem = cnefe_segmento`;
- segmento com CEP dos Correios **e** vencedor: igual → `confirmado_cnefe` (confiança `alta`); diferente e `alta`/`media` → troca (`cep_origem = cnefe_segmento`, `cnefe_motivo = correios_divergente:<cep dos correios>`); diferente e `baixa` → mantém (`correios_divergente_fraco:<cep do cnefe>`);
- `multi_cep` **sem** vencedor → volta aos CEPs dos Correios (`correios_geosanja:multi_cep_sem_cnefe`, confiança `baixa`);
- `ambiguo`, `sem_cep`, `sem_nome` e suspeitos **sem** vencedor → ficam sem CEP;
- segmento sem nome com vencedor → recebe o nome (e o tipo, se faltar) do CNEFE;
- `cep_entre_candidatos`: nos `multi_cep`/`ambiguo`, `Sim` se o vencedor estava entre os CEPs dos Correios;
- `obs_cep`: texto curto com o que aconteceu.

In [ ]:
log.info("Aplicando os resultados na base...")
col_venc = venc.rename(columns={"id_seg": "id", "cep": "v_cep", "confianca": "v_confianca", "n_locais": "cnefe_n_locais",
                                "participacao": "cnefe_participacao", "tipo_cnefe": "v_tipo", "nome_cnefe": "v_nome"})
col_venc = col_venc[["id", "v_cep", "v_confianca", "cnefe_n_locais", "cnefe_participacao", "cnefe_votos", "v_tipo", "v_nome"]]
if pd.api.types.is_integer_dtype(base["id"]):
    col_venc["id"] = col_venc["id"].astype(base["id"].dtype)                       # mesmo tipo para a junção
base = base.merge(col_venc, on="id", how="left", validate="one_to_one")
assert len(base) == n_orig, "A junção mudou o número de feições!"

aberto = base["cnefe_motivo"] != ""                                               # segmento sem CEP antes da votação
com_correios = (base["cep"] != "") & ~aberto                                       # segmento com CEP único dos correios
confirmado = com_correios & (base["v_cep"] == base["cep"])                         # cnefe concorda com os correios
divergente = com_correios & base["v_cep"].notna() & (base["v_cep"] != base["cep"])
troca = divergente & base["v_confianca"].isin(["alta", "media"])                   # cnefe corrige os correios
fraco = divergente & ~troca                                                        # cnefe discorda, mas com votação fraca
base.loc[confirmado, "cep_confianca"] = "alta"
base.loc[confirmado, "cnefe_motivo"] = "confirmado_cnefe"
base.loc[troca, "cnefe_motivo"] = "correios_divergente:" + base.loc[troca, "cep_correios"]
base.loc[fraco, "cnefe_motivo"] = "correios_divergente_fraco:" + base.loc[fraco, "v_cep"]
log.info("Segmentos com CEP dos correios: %d | confirmados pelo cnefe: %d | trocados (alta/média): %d | divergência fraca (mantidos): %d | sem pontos: %d",
         int(com_correios.sum()), int(confirmado.sum()), int(troca.sum()), int(fraco.sum()),
         int((com_correios & base["v_cep"].isna()).sum()))

tem = (aberto & base["v_cep"].notna()) | troca                                     # segmento que recebe o CEP do CNEFE
base.loc[tem, "cep"] = base.loc[tem, "v_cep"]
base.loc[tem, "cep_origem"] = "cnefe_segmento"
base.loc[tem, "cep_confianca"] = base.loc[tem, "v_confianca"]

multi_sem = (base["status_cep"] == "multi_cep") & ~tem                             # multi_cep sem pontos: volta aos correios
base.loc[multi_sem, "cep"] = base.loc[multi_sem, "cep_correios"]
base.loc[multi_sem, "cep_origem"] = "correios_geosanja:multi_cep_sem_cnefe"
base.loc[multi_sem, "cep_confianca"] = "baixa"

# --- O vencedor estava entre os CEPs dos correios? (só multi_cep e ambíguo) ------------------------------------------
ref = np.where(base["status_cep"] == "ambiguo", base["ceps_candidatos"], base["cep_correios"])
entre = [("Sim" if c in r.split("|") else "Não") if isinstance(c, str) and r else "" for c, r in zip(base["v_cep"], ref)]
base["cep_entre_candidatos"] = np.where(tem & base["status_cep"].isin(["multi_cep", "ambiguo"]), entre, "")

# --- Nome do CNEFE nos segmentos sem nome --------------------------------------------------------------------------------
base["nome_geosanja"] = base["nome"]                                               # nome original, para conferência
recebe_nome = tem & base["seg_sem_nome"] & base["v_nome"].notna()
base.loc[recebe_nome, "nome"] = base.loc[recebe_nome, "v_nome"]
sem_tipo = recebe_nome & (base["tipo"] == "")
base.loc[sem_tipo, "tipo"] = base.loc[sem_tipo, "v_tipo"]
base["nome_origem"] = np.select([recebe_nome, base["nome"] != ""], ["cnefe", "geosanja"], default="")

# --- Observação curta -------------------------------------------------------------------------------------------------------
part_txt = (base["cnefe_participacao"] * 100).round(0).astype("Int64").astype(str)
base["obs_cep"] = np.select(
    [troca,
     confirmado,
     fraco,
     tem,
     multi_sem,
     (base["cnefe_motivo"] != "") & ~tem],
    ["CNEFE trocou o CEP dos correios " + base["cep_correios"] + ": " + base["cnefe_n_locais"].astype("Int64").astype(str)
     + " locais, " + part_txt + "% dos votos",
     "CNEFE confirmou o CEP dos correios: " + base["cnefe_n_locais"].astype("Int64").astype(str) + " locais, " + part_txt + "% dos votos",
     "CNEFE indica " + base["v_cep"].fillna("") + ", mas com votação fraca (" + part_txt + "%): mantido o CEP dos correios",
     "CNEFE: " + base["cnefe_n_locais"].astype("Int64").astype(str) + " locais, " + part_txt + "% dos votos ("
     + base["cnefe_motivo"] + ")",
     "Sem pontos do CNEFE: mantidos os CEPs dos correios",
     "Sem pontos do CNEFE no trecho (" + base["cnefe_motivo"] + ")"],
    default="")
base["cep_conferido"] = "Não"

log.info("Segmentos que receberam nome do cnefe: %d", int(recebe_nome.sum()))
log.info("Origem do CEP:\n%s", base["cep_origem"].replace("", "(sem CEP)").value_counts().to_string())
log.info("Confiança:\n%s", base["cep_confianca"].replace("", "(sem CEP)").value_counts().to_string())
log.info("Base atualizada!")

## 10. Exportação
- **Base:** campos originais + colunas de CEP. Se a base editada já existir, grava só a versão automática em `03_processamento` (não sobrescreve as suas edições).
- **Locais do CNEFE** (`cnefe_locais.parquet` e `pontos_cnefe.gpkg`): sempre regravados.
- **Relatório** (`relatorio_base_cnefe.xlsx`): Resumo · CEPs suspeitos (retirados dos segmentos) · Segmentos pelo CNEFE.

In [ ]:
log.info("Gravando a base...")
colunas_cep = ["status_cep", "cep", "cep_origem", "cep_confianca", "cep_conferido", "cep_correios", "ceps_candidatos",
               "cnefe_motivo", "cnefe_votos", "cnefe_n_locais", "cnefe_participacao", "cep_entre_candidatos",
               "nome_origem", "nome_geosanja", "obs_cep"]
saida = gpd.GeoDataFrame(base[colunas_originais + colunas_cep], geometry=base.geometry, crs=base.crs)
for col in colunas_cep + ["tipo", "nome"]:
    if not pd.api.types.is_numeric_dtype(saida[col]):                                     # colunas de texto
        texto = saida[col].astype(object)
        saida[col] = texto.where(texto.notna() & (texto.astype(str) != ""), None)          # vazio vira nulo no GeoPackage

if ARQ_BASE.exists():                                            # a base editada já existe: NUNCA é sobrescrita
    destino = PASTA_SAIDA / "logradouros_cep_automatico.gpkg"
    log.warning("A base editada %s já existe e NÃO foi alterada.", ARQ_BASE.name)
    log.warning("Versão automática gravada só para comparação em: %s", destino)
    if destino.exists():
        destino.unlink()                                         # a versão automática é regerável
else:
    destino = ARQ_BASE                                           # primeira vez: cria a base em 02_dados_editados
    destino.parent.mkdir(parents=True, exist_ok=True)
    log.info("A base editada ainda não existe: criando em %s", destino)
assert len(saida) == n_orig, "Perda de feições na exportação!"
para_gpkg(saida).to_file(destino, layer=CAMADA_BASE, driver="GPKG", engine="pyogrio")
log.info("Base salva: %s (%d feições, CRS %s)!", destino.name, len(saida), saida.crs)
validar_tabela("Base de logradouros com CEP", saida, chave=["id"], obrigatorias=["id", "status_cep"], linhas_entrada=n_orig)

log.info("Gravando os locais do cnefe...")
locais["x"] = locais.geometry.x                                  # coordenadas em metros (crs_metrico)
locais["y"] = locais.geometry.y
col_loc = ["loc_id", "cep", "lat", "lon", "x", "y", "n_enderecos", "tipo_cnefe", "nome_cnefe", "id_seg", "dist", "ligacao"]
pd.DataFrame(locais[col_loc]).to_parquet(ARQ_LOCAIS, index=False)
if ARQ_PONTOS.exists():
    ARQ_PONTOS.unlink()
pts = locais[col_loc + ["geometry"]].copy()
pts["dist"] = pts["dist"].round(1)
para_gpkg(pts).to_file(ARQ_PONTOS, layer="pontos_cnefe", driver="GPKG", engine="pyogrio")
log.info("Locais salvos: %s e %s (%d locais)!", ARQ_LOCAIS.name, ARQ_PONTOS.name, len(locais))

log.info("Gravando o relatório...")
resumo = pd.concat([
    saida["status_cep"].value_counts().rename_axis("valor").reset_index(name="segmentos").assign(item="status da etapa 02"),
    saida["cep_origem"].fillna("(sem CEP)").value_counts().rename_axis("valor").reset_index(name="segmentos").assign(item="cep_origem"),
    saida["cep_confianca"].fillna("(sem CEP)").value_counts().rename_axis("valor").reset_index(name="segmentos").assign(item="cep_confianca"),
])[["item", "valor", "segmentos"]]
aba_susp = aud[aud["suspeito"]].sort_values("n_locais", ascending=False).drop(columns="suspeito")
aba_seg = pd.DataFrame(saida.loc[saida["cep_origem"] == "cnefe_segmento",
                                 ["id", "tipo", "nome", "loteamento", "status_cep", "cep_correios", "ceps_candidatos", "cep",
                                  "cep_confianca", "cnefe_motivo", "cnefe_votos", "cnefe_n_locais", "cnefe_participacao",
                                  "cep_entre_candidatos", "nome_origem", "nome_geosanja"]])
with pd.ExcelWriter(ARQ_EXCEL, engine="openpyxl") as escritor:
    for nome_aba, tabela in [("Resumo", resumo), ("CEPs suspeitos", aba_susp), ("Segmentos pelo CNEFE", aba_seg)]:
        tabela.to_excel(escritor, sheet_name=nome_aba, index=False)
        formatar_excel(escritor, nome_aba)
log.info("Relatório salvo: %s!", ARQ_EXCEL.name)
log.info("Exportação concluída!")

## 11. Resumo

In [ ]:
log.info("Resumo final...")
nomeados = int((saida["status_cep"] != "sem_nome").sum())
com_cep = saida["cep"].notna()
log.info("Segmentos: %d | com CEP: %d (%.1f%%) | sem CEP: %d", len(saida), int(com_cep.sum()), com_cep.mean() * 100, int((~com_cep).sum()))
log.info("  pelos correios: %d | pelo cnefe: %d | multi_cep sem cnefe: %d",
         int(saida["cep_origem"].fillna("").str.match(r"^correios_geosanja:(?!multi)").sum()),
         int((saida["cep_origem"] == "cnefe_segmento").sum()),
         int((saida["cep_origem"] == "correios_geosanja:multi_cep_sem_cnefe").sum()))
log.info("  CEPs dos correios retirados por suspeita: %d (segmentos liberados: %d)", len(ceps_suspeitos), int(mask_s.sum()))
log.info("  segmentos dos correios confirmados pelo cnefe: %d | trocados: %d | divergência fraca (mantidos): %d",
         int(confirmado.sum()), int(troca.sum()), int(fraco.sum()))
log.info("  segmentos sem nome que ganharam nome do cnefe: %d", int((saida["nome_origem"] == "cnefe").sum()))
log.info("Sem CEP por status da etapa 02:\n%s", saida.loc[~com_cep, "status_cep"].value_counts().to_string())
log.info("Próximo: rodar as etapas 04, 05 e 06.")
log.info("Etapa 03 concluída!")